# Dataset

## Download Dataset

In [1]:
#!wget -O traffic_buddy_train+public_test.zip "https://dl-challenge.zalo.ai/2025/TrafficBuddyHZUTAXF/traffic_buddy_train+public_test.zip"

In [2]:
#!unzip -q traffic_buddy_train+public_test.zip

In [3]:
#!rm -rf traffic_buddy_train+public_test.zip

## Download Video Data

In [4]:
%cd traffic_buddy_train+public_test

/workspace/traffic_buddy_train+public_test


## Read Data anf build Dataframe

In [5]:
# Cai dat moi truong - tuong tu Colab mac dinh (khong pin version cu the)
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu124
%pip install opencv-python pandas Pillow transformers datasets matplotlib tqdm
# Cai FlashAttention2 de tang toc do training (compile time co the lau)
%pip install flash-attn --no-build-isolation


Looking in indexes: https://download.pytorch.org/whl/cu124

[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: /root/.venv/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: /root/.venv/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: /root/.venv/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [6]:
# ============================================================
# SECTION: Import thu vien va dinh nghia cac ham/class con thieu
# ============================================================
import os
import sys
import math
import json
import cv2
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torchvision.transforms as T
from torchvision.transforms import InterpolationMode
from torch.utils.data import Dataset
import matplotlib.pyplot as plt
import warnings
import types
from tqdm import tqdm
from datasets import Dataset as HFDataset
from transformers import AutoModel, AutoTokenizer, TrainingArguments, Trainer

os.environ["TORCH_COMPILE_DISABLE"] = "1"

print(f"PyTorch version: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")

print(sys.executable)



/root/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch version: 2.6.0+cu124
GPU available: True
GPU device: Tesla V100-SXM2-32GB
CUDA version: 12.4
/root/.venv/bin/python


In [7]:
import pandas as pd
train_df = pd.read_json('train/train.json')
# print(train_df)
train_df = pd.json_normalize(train_df['data'])
display(train_df.head())

,id,question,choices,answer,support_frames,video_path,_unused_
0,train_0001,Nếu xe ô tô đang chạy ở làn ngoài cùng bên phả...,"[A. Đúng, B. Sai]",B. Sai,[4.427402],train/videos/2b840c67_386_clip_002_0008_0018_Y...,cbb77f7bf70be7d60ac580753f67ee61@1
1,train_0002,Phần đường trong video cho phép các phương tiệ...,"[A. Đi thẳng, B. Đi thẳng và rẽ phải, C. Đi th...","C. Đi thẳng, rẽ trái và rẽ phải",[5.344766],train/videos/2b840c67_386_clip_002_0008_0018_Y...,cbb77f7bf70be7d60ac580753f67ee61@2
2,train_0003,Biển chỉ dẫn 3 hướng di chuyển chính tiếp theo...,"[A. Đúng, B. Sai]",A. Đúng,[3.845463],train/videos/fe716b14_386_clip_003_0018_0024_N...,cbb77f7bf70be7d60ac580753f67ee61@3
3,train_0004,"Theo biển báo trong video, muốn đi đến đường L...","[A. Đi thẳng, B. Rẽ trái, C. Rẽ phải, D. Không...",B. Rẽ trái,[7.238903],train/videos/d02ce0c1_386_clip_004_0024_0032_N...,cbb77f7bf70be7d60ac580753f67ee61@4
4,train_0005,Làn đường di chuyển hiện tại có được rẽ phải k...,"[A. Có, B. Không]",B. Không,[1.359896],train/videos/2339ba19_386_clip_005_0032_0039_Y...,cbb77f7bf70be7d60ac580753f67ee61@5


In [8]:
test_df = pd.read_json('public_test/public_test.json')
test_df = pd.json_normalize(test_df['data'])
display(test_df.head())

,id,question,choices,video_path
0,testa_0001,"Theo trong video, nếu ô tô đi hướng chếch sang...","[A. Không có thông tin, B. Dầu Giây Long Thành...",public_test/videos/efc9909e_908_clip_001_0000_...
1,testa_0002,"Theo trong video, nếu ô tô đi hướng chếch sang...","[A. Đúng, B. Sai]",public_test/videos/efc9909e_908_clip_001_0000_...
2,testa_0003,Trong video có xuất hiện biển cảnh báo không?,"[A. Có, B. Không]",public_test/videos/7194eae2_908_clip_004_0020_...
3,testa_0004,Phía trước có đèn giao thông không?,"[A. Có, B. Không]",public_test/videos/37f8a88b_884_clip_001_0000_...
4,testa_0005,"Trong video, biển báo cấm ở góc phía bên phải ...","[A. Xe đạp, B. Xe máy, C. Xe container, D. Xe ...",public_test/videos/37f8a88b_884_clip_001_0000_...


In [9]:
test_df[:1].to_csv("abc")

## Training

In [10]:
# ============================================================
# IMPLEMENT: prepare_model_for_training
# Chuan bi model cho qua trinh fine-tuning
# ============================================================
def prepare_model_for_training(model, use_gradient_checkpointing=True):
    """Chuan bi model cho training:
    1. Chuyen sang train mode
    2. Enable gradients cho tat ca parameters
    3. Bat gradient checkpointing de giam memory
    """
    model.train()
    for param in model.parameters():
        param.requires_grad = True
    if use_gradient_checkpointing:
        if getattr(model, 'supports_gradient_checkpointing', False):
            model.gradient_checkpointing_enable()
            print("[INFO] Gradient checkpointing enabled")
        else:
            print("[WARN] Model does not support gradient checkpointing, skipping")
    print("[INFO] Model prepared for training")
    print(f'  - Total params: {sum(p.numel() for p in model.parameters()):,}')
    print(f'  - Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')
    return model


In [11]:
# ============================================================
# IMPLEMENT: setup_training
# Setup model va tokenizer cho training
# ============================================================
def setup_training(model_name="5CD-AI/Vintern-1B-v3_5"):
    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        trust_remote_code=True,
        use_fast=False
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # transformers v5.x compatibility: patch missing all_tied_weights_keys
    from transformers.dynamic_module_utils import get_class_from_dynamic_module
    try:
        InternVLChatModel = get_class_from_dynamic_module(
            "modeling_internvl_chat.InternVLChatModel", model_name
        )
        if not hasattr(InternVLChatModel, '_tied_weights_keys'):
            InternVLChatModel._tied_weights_keys = []
    except Exception:
        pass

    # Load model
    try:
        model = AutoModel.from_pretrained(
            model_name,
            torch_dtype=torch.bfloat16,
            low_cpu_mem_usage=True,
            trust_remote_code=True,
            use_flash_attn=True,
        ).eval()
    except Exception:
        model = AutoModel.from_pretrained(
            model_name,
            torch_dtype=torch.bfloat16,
            low_cpu_mem_usage=True,
            trust_remote_code=True
        ).eval()

    # transformers v5.x safety: ensure all_tied_weights_keys exists
    if not hasattr(model, 'all_tied_weights_keys'):
        model.all_tied_weights_keys = {}

    # Prepare for training
    model = prepare_model_for_training(model)


    # ---- Fix: set img_context_token_id cho model (can thiet cho forward/generate) ----
    model.img_context_token_id = tokenizer.convert_tokens_to_ids('<IMG_CONTEXT>')

    return model, tokenizer


In [12]:
# ============================================================
# IMPLEMENT: TrafficVideoQADataset
# Dataset class cho Traffic Video QA task
# ============================================================

# Constants cho image preprocessing (ImageNet stats)
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

def build_transform(input_size):
    """Tao transformation pipeline: RGB -> Resize -> Tensor -> Normalize"""
    return T.Compose([
        T.Lambda(lambda img: img.convert('RGB') if img.mode != 'RGB' else img),
        T.Resize((input_size, input_size), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])

def extract_single_frame(video_path, frame_time, input_size=448):
    """Trich xuat 1 frame tu video tai thoi diem frame_time (giay)"""
    if not os.path.exists(video_path):
        print(f"[WARN] Video not found: {video_path}")
        return None
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"[WARN] Cannot open video: {video_path}")
        return None
    try:
        fps = cap.get(cv2.CAP_PROP_FPS)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if fps is None or fps <= 0 or math.isnan(fps):
            fps = 30.0
        frame_index = int(frame_time * fps)
        frame_index = max(0, min(frame_index, max(0, total_frames - 1)))
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_index)
        ret, frame = cap.read()
        if not ret:
            cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
            ret, frame = cap.read()
        if not ret:
            return None
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        pil_image = Image.fromarray(frame_rgb)
        pixel_values = build_transform(input_size)(pil_image)
        return pixel_values
    finally:
        cap.release()

class TrafficVideoQADataset(Dataset):
    """Dataset cho Traffic Video QA.
    Moi item tra ve:
        - input_ids: token ids cua prompt (question + choices + answer)
        - labels: input_ids voi -100 o phan question (chi tinh loss tren answer)
        - pixel_values: frame da duoc transform (3, H, W)
        - image_flags: vector danh dau vi tri <image> token
    """
    def __init__(self, dataframe, video_base_path, tokenizer, input_size=448, max_length=1024):
        self.df = dataframe.reset_index(drop=True)
        self.video_base_path = video_base_path
        self.tokenizer = tokenizer
        self.input_size = input_size
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        sample = self.df.iloc[idx]
        video_path = sample.get('video_path', '')
        question = sample.get('question', '')
        choices = sample.get('choices', [])
        answer = sample.get('answer', '')
        support_frames = sample.get('support_frames', [])
        frame_time = support_frames[0] if support_frames else 0.0
        full_video_path = os.path.join(self.video_base_path, video_path)
        pixel_values = extract_single_frame(full_video_path, frame_time, self.input_size)
        if pixel_values is None:
            pixel_values = torch.zeros(3, self.input_size, self.input_size)
        # ---- Fix: prompt phai dung <img> + 256 x <IMG_CONTEXT> + </img> ----
        NUM_IMAGE_TOKEN = 256  # (448/14)^2 * 0.5^2
        image_tokens = "<img>" + "<IMG_CONTEXT>" * NUM_IMAGE_TOKEN + "</img>"
        if choices and isinstance(choices, list):
            choices_str = "\n".join(choices)
            text_input = f"{question}\n{choices_str}"
        else:
            text_input = question
        prompt = f"{image_tokens}\nHuman: {text_input}\nAssistant: {answer}"
        tokenized = self.tokenizer(
            prompt, padding=False, truncation=True,
            max_length=self.max_length, return_tensors='pt',
            add_special_tokens=True
        )
        input_ids = tokenized['input_ids'][0]
        # Labels: mask phan question bang -100, chi tinh loss tren answer
        labels = input_ids.clone()
        assistant_pos = prompt.find("Assistant:")
        if assistant_pos != -1:
            prefix = prompt[:assistant_pos + len("Assistant:")]
            prefix_tokens = self.tokenizer(prefix, add_special_tokens=False).input_ids
            prefix_len = len(prefix_tokens) if isinstance(prefix_tokens, list) else prefix_tokens.shape[0]
            if prefix_len < len(labels):
                labels[:prefix_len] = -100
        # ---- Fix: image_flags la per-sample flag (khong phai per-token mask) ----
        # Model can: vit_embeds[image_flags == 1], voi image_flags shape [batch]
        # Moi sample trong dataset nay deu co anh, nen flag = 1
        image_flags = torch.ones(1, dtype=torch.long)
        # OLD: image_token_id = self.tokenizer.convert_tokens_to_ids('<image>')
        # OLD: image_flags = torch.zeros_like(input_ids, dtype=torch.long)
        # OLD: for i, tid in enumerate(input_ids):
        # OLD:     if tid.item() == image_token_id:
        # OLD:         image_flags[i] = 1
        return {
            'input_ids': input_ids,
            'labels': labels,
            'pixel_values': pixel_values,
            'image_flags': image_flags,
        }


In [13]:
# ============================================================
# IMPLEMENT: TrafficVideoQACollator
# Collator de pad cac sample thanh batch cho Trainer
# ============================================================
class TrafficVideoQACollator:
    """Collator: pad input_ids, labels (-100), attention_mask, pixel_values, image_flags"""
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
        self.pad_token_id = tokenizer.pad_token_id or tokenizer.eos_token_id

    def __call__(self, batch):
        input_ids_list = [item['input_ids'] for item in batch]
        max_len = max(len(ids) for ids in input_ids_list)
        # Pad input_ids + attention_mask
        padded_input_ids, attention_masks = [], []
        for ids in input_ids_list:
            seq_len = len(ids)
            pad_len = max_len - seq_len
            padded = torch.cat([ids, torch.full((pad_len,), self.pad_token_id, dtype=ids.dtype)])
            padded_input_ids.append(padded)
            mask = torch.cat([torch.ones(seq_len, dtype=torch.long), torch.zeros(pad_len, dtype=torch.long)])
            attention_masks.append(mask)
        input_ids = torch.stack(padded_input_ids)
        attention_mask = torch.stack(attention_masks)
        # Pad labels voi -100
        labels_list = [item['labels'] for item in batch]
        padded_labels = []
        for lbs in labels_list:
            seq_len = len(lbs)
            pad_len = max_len - seq_len
            padded = torch.cat([lbs, torch.full((pad_len,), -100, dtype=lbs.dtype)])
            padded_labels.append(padded)
        labels = torch.stack(padded_labels)
        # Stack pixel_values; image_flags la per-sample flag [1], chi can stack
        pixel_values = torch.stack([item['pixel_values'] for item in batch])
        # OLD: flags_list = [item['image_flags'] for item in batch]
        # OLD: padded_flags = []
        # OLD: for fl in flags_list:
        # OLD:     seq_len = len(fl)
        # OLD:     pad_len = max_len - seq_len
        # OLD:     padded = torch.cat([fl, torch.zeros(pad_len, dtype=fl.dtype)])
        # OLD:     padded_flags.append(padded)
        # Fix: image_flags shape [batch, 1], chi stack khong pad
        image_flags = torch.stack([item['image_flags'] for item in batch])
        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': labels,
            'pixel_values': pixel_values,
            'image_flags': image_flags,
        }


In [14]:
def train_model(train_df, val_df, video_base_path="", output_dir="./vintern-traffic-qa"):
    # Setup model and tokenizer
    model, tokenizer = setup_training()
    model = model.cuda()

    # Create datasets
    train_dataset = TrafficVideoQADataset(train_df, video_base_path, tokenizer)
    val_dataset = TrafficVideoQADataset(val_df, video_base_path, tokenizer)

    # Create collator
    data_collator = TrafficVideoQACollator(tokenizer)

    # Training arguments
    training_args = TrainingArguments(
        output_dir=output_dir,
        overwrite_output_dir=True,
        num_train_epochs=3,
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=100,
        learning_rate=2e-5,
        bf16=True,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=100,
        save_steps=500,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        report_to=None,
        dataloader_pin_memory=False,
        remove_unused_columns=False,
    )

    # Create trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=data_collator,
        processing_class=tokenizer,
    )

    # Start training
    print("Starting training...")
    trainer.train()

    # Save final model
    trainer.save_model(f"{output_dir}-final")
    tokenizer.save_pretrained(f"{output_dir}-final")

    print(f"Training complete! Model saved to {output_dir}-final")

    return model, tokenizer

In [15]:
train_size = int(0.8 * len(train_df))
train_subset = train_df[:train_size]
val_subset = train_df[train_size:]

In [16]:
model, tokenizer = train_model(
    train_subset,
    val_subset,
    video_base_path="",
    output_dir="./vintern-traffic-qa"
)

/root/.venv/lib/python3.12/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
`torch_dtype` is deprecated! Use `dtype` instead!


FlashAttention2 is not installed.
[WARN] Model does not support gradient checkpointing, skipping
[INFO] Model prepared for training
  - Total params: 938,193,024
  - Trainable params: 938,193,024


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 151645, 'pad_token_id': 151643}.


Starting training...


/root/.venv/lib/python3.12/site-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss
100,0.141300,0.084935
200,0.151200,0.086603
300,0.112600,0.075216
400,0.049700,0.092148


Training complete! Model saved to ./vintern-traffic-qa-final


In [28]:
# Load your model
model = AutoModel.from_pretrained(
    "./vintern-traffic-qa-final",
    trust_remote_code=True
).eval().cuda()
model.img_context_token_id = tokenizer.convert_tokens_to_ids('<IMG_CONTEXT>')
tokenizer = AutoTokenizer.from_pretrained("./vintern-traffic-qa-final",fix_mistral_regex=True)


In [29]:
# Test dataframe
test_df.shape

(405, 4)

In [30]:
def predict_and_save_csv(model, tokenizer, test_df, video_base_path, output_csv, input_size=448, max_length=1024):
    model.eval()
    results = []
    print(f"Running inference on {len(test_df)} test samples...")
    for idx in tqdm(range(len(test_df)), desc="Predicting"):
        sample = test_df.iloc[idx]
        sample_id = sample.get('id', f'unknown_{idx}')
        question = sample.get('question', '')
        choices = sample.get('choices', [])
        video_path = sample.get('video_path', '')
        support_frames = sample.get('support_frames', [])
        frame_time = support_frames[0] if support_frames else 0.0
        try:
            full_video_path = os.path.join(video_base_path, video_path)
            pixel_values = extract_single_frame(full_video_path, frame_time, input_size)
            if pixel_values is None:
                pixel_values = torch.zeros(3, input_size, input_size)
            pixel_values = pixel_values.unsqueeze(0).cuda()

            NUM_IMAGE_TOKEN = 256
            image_tokens = "<img>" + "<IMG_CONTEXT>" * NUM_IMAGE_TOKEN + "</img>"
            if choices and isinstance(choices, list):
                choices_str = "\n".join(choices)
                text_input = f"{question}\n{choices_str}"
            else:
                text_input = question
            prompt = f"{image_tokens}\nHuman: {text_input}\nAssistant:"

            tokenized = tokenizer(
                prompt, padding=True, truncation=True,
                max_length=max_length, return_tensors='pt',
                add_special_tokens=True
            )
            input_ids = tokenized['input_ids'].cuda()
            attention_mask = tokenized['attention_mask'].cuda()

            with torch.no_grad():
                generated_ids = model.generate(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    pixel_values=pixel_values,
                    max_new_tokens=64, num_beams=1,
                    do_sample=False,
                    pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                )

            input_len = input_ids.shape[1]
            generated_part = generated_ids[0][input_len:]
            answer = tokenizer.decode(generated_part, skip_special_tokens=True).strip()
            if not answer:
                answer = "N/A"
            results.append({'id': sample_id, 'answer': answer})

        except Exception as e:
            import traceback
            print(f"[ERROR] Sample {sample_id}: {e}")
            traceback.print_exc()
            results.append({'id': sample_id, 'answer': 'N/A'})

    result_df = pd.DataFrame(results)
    result_df.to_csv(output_csv, index=False, encoding='utf-8-sig')
    print(f"Submission saved to {output_csv}")
    print(f"Total samples: {len(results)}")
    print(result_df.head())
    return result_df


In [31]:
# Run prediction
predict_and_save_csv(model, tokenizer, test_df, "", "submission.csv")


Running inference on 405 test samples...


Predicting: 100%|██████████| 405/405 [03:51<00:00,  1.75it/s]

Submission saved to submission.csv
Total samples: 405
           id answer
0  testa_0001    N/A
1  testa_0002    N/A
2  testa_0003    N/A
3  testa_0004    N/A
4  testa_0005    N/A


,id,answer
0,testa_0001,N/A
1,testa_0002,N/A
2,testa_0003,N/A
3,testa_0004,N/A
4,testa_0005,N/A
...,...,...
400,testa_0401,N/A
401,testa_0402,N/A
402,testa_0403,N/A
403,testa_0404,N/A


# EDA

In [32]:
import os
import json
import pandas as pd
import numpy as np
import cv2
from PIL import Image
import torch
import matplotlib.pyplot as plt
import math
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModel, TrainingArguments, Trainer
import torchvision.transforms as T
from torchvision.transforms import InterpolationMode
import warnings
import types
from datasets import Dataset
warnings.filterwarnings("ignore", message="torch.utils.checkpoint")
warnings.filterwarnings("ignore", message="None of the inputs have requires_grad=True")

In [33]:
# Constants from author's code
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

def build_transform(input_size):
    MEAN, STD = IMAGENET_MEAN, IMAGENET_STD
    transform = T.Compose([
        T.Lambda(lambda img: img.convert('RGB') if img.mode != 'RGB' else img),
        T.Resize((input_size, input_size), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=MEAN, std=STD)
    ])
    return transform

In [34]:
def extract_frame_from_video(video_path: str, frame_time: float, input_size=448):
    """Extract single frame from video and return both original and processed versions"""
    print(f"Extracting frame from: {video_path}")
    print(f"Frame time: {frame_time}s")

    if not os.path.exists(video_path):
        print(f"❌ Video file not found: {video_path}")
        return None, None

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"❌ Cannot open video: {video_path}")
        cap.release()
        return None, None

    try:
        fps = cap.get(cv2.CAP_PROP_FPS)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        duration = total_frames / fps if fps > 0 else 0

        print(f"Video info - FPS: {fps:.2f}, Total frames: {total_frames}, Duration: {duration:.2f}s")

        if fps is None or fps <= 0 or math.isnan(fps):
            print("⚠️ Invalid FPS, using first frame")
            cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
            ret, frame = cap.read()
            if ret:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                original_image = Image.fromarray(frame)
                processed_image = original_image.resize((input_size, input_size))
                return original_image, processed_image
            return None, None

        frame_index = int(frame_time * fps)
        frame_index = max(0, min(frame_index, max(0, total_frames-1)))

        print(f"Target frame index: {frame_index}/{total_frames-1}")

        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_index)
        ret, frame = cap.read()

        if not ret:
            print("⚠️ Cannot read target frame, trying first frame")
            cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
            ret, frame = cap.read()

        if ret:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            original_image = Image.fromarray(frame)
            processed_image = original_image.resize((input_size, input_size))
            print(f"✅ Successfully extracted frame {frame_index}")
            return original_image, processed_image
        else:
            print("❌ Failed to extract any frame")
            return None, None
    finally:
        cap.release()

In [35]:
def display_sample_data(sample_idx=0):
    """Display detailed information about one sample"""

    # Load tokenizer
    print("🔧 Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained("5CD-AI/Vintern-1B-v3_5", trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Load data
    print("\n📊 Loading data...")
    try:
        with open('train/train.json', 'r', encoding='utf-8') as f:
            data = json.load(f)

        if 'data' in data:
            df = pd.DataFrame(data['data'])
        else:
            df = pd.DataFrame(data)

        print(f"✅ Loaded {len(df)} samples from train.json")
    except Exception as e:
        print(f"❌ Error loading data: {e}")
        # Create sample data for testing
        df = pd.DataFrame([{
            'id': 'train_0001',
            'question': 'Nếu xe ô tô đang chạy ở làn ngoài cùng bên phải trong video này thì xe đó chỉ được phép rẽ phải?',
            'choices': ['A. Đúng', 'B. Sai'],
            'answer': 'B. Sai',
            'support_frames': [4.427402],
            'video_path': 'train/videos/2b840c67_386_clip_002_0008_0018_Y.mp4'
        }])
        print("⚠️ Using sample data for demonstration")

    # Get sample
    if sample_idx >= len(df):
        print(f"❌ Sample index {sample_idx} out of range. Using first sample.")
        sample_idx = 0

    sample = df.iloc[sample_idx]

    print(f"\n{'='*80}")
    print(f"📋 SAMPLE DATA ANALYSIS - Index {sample_idx}")
    print(f"{'='*80}")

    # Display all sample information
    print(f"📁 Sample ID: {sample.get('id', 'N/A')}")
    print(f"❓ Question: {sample.get('question', 'N/A')}")

    choices = sample.get('choices', [])
    if isinstance(choices, list):
        print(f"🔘 Choices:")
        for i, choice in enumerate(choices):
            print(f"   {i+1}. {choice}")
    else:
        print(f"🔘 Choices: {choices}")

    print(f"✅ Answer: {sample.get('answer', 'N/A')}")
    print(f"🎞️ Support frames: {sample.get('support_frames', 'N/A')}")
    print(f"📹 Video path: {sample.get('video_path', 'N/A')}")

    # Check if video file exists
    video_path = sample.get('video_path', '')
    full_video_path = os.path.join('', video_path)  # video_base_path is empty in your code

    print(f"\n🔍 Checking video file...")
    print(f"Relative path: {video_path}")
    print(f"Full path: {full_video_path}")
    print(f"File exists: {os.path.exists(full_video_path)}")

    if os.path.exists(full_video_path):
        # Get video info
        cap = cv2.VideoCapture(full_video_path)
        fps = cap.get(cv2.CAP_PROP_FPS)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        duration = total_frames / fps if fps > 0 else 0
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        cap.release()

        print(f"🎥 Video details:")
        print(f"   - Resolution: {width}x{height}")
        print(f"   - FPS: {fps:.2f}")
        print(f"   - Total frames: {total_frames}")
        print(f"   - Duration: {duration:.2f}s")
    else:
        print("❌ Video file not found!")

    # Extract and display frame
    print(f"\n🖼️ Extracting frame...")
    support_frames = sample.get('support_frames', [])
    frame_time = support_frames[0] if support_frames else 0.0

    original_img, processed_img = extract_frame_from_video(full_video_path, frame_time)

    # Display images
    if original_img is not None and processed_img is not None:
        print(f"\n📸 Displaying images...")

        # Create figure with subplots
        fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 6))

        # Original image
        ax1.imshow(original_img)
        ax1.set_title(f'Original Frame\nTime: {frame_time}s\nSize: {original_img.size}', fontsize=12)
        ax1.axis('off')

        # Processed image
        ax2.imshow(processed_img)
        ax2.set_title(f'Processed Frame\nSize: {processed_img.size}', fontsize=12)
        ax2.axis('off')

        # Apply transform to see normalized version
        transform = build_transform(448)
        tensor_img = transform(processed_img)

        # Convert back to displayable format (denormalize)
        mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
        std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
        denormalized = tensor_img * std + mean
        denormalized = torch.clamp(denormalized, 0, 1)
        display_img = T.ToPILImage()(denormalized)

        ax3.imshow(display_img)
        ax3.set_title(f'After Transform\n(Normalized & Processed)', fontsize=12)
        ax3.axis('off')

        plt.tight_layout()
        plt.show()

        # Print tensor information
        print(f"\n🎯 Tensor information:")
        print(f"   - Shape: {tensor_img.shape}")
        print(f"   - Data type: {tensor_img.dtype}")
        print(f"   - Value range: [{tensor_img.min():.3f}, {tensor_img.max():.3f}]")
        print(f"   - Mean per channel: {tensor_img.mean(dim=(1, 2))}")
        print(f"   - Std per channel: {tensor_img.std(dim=(1, 2))}")

    # Tokenization demonstration
    print(f"\n🔤 Tokenization analysis:")
    question = sample['question']
    choices = "\n".join(sample['choices']) if isinstance(sample['choices'], list) else sample.get('choices', '')
    answer = sample['answer']

    if choices:
        text_input = f"{question}\n{choices}"
    else:
        text_input = question

    # Fix: expand <image> thanh <img> + 256 x <IMG_CONTEXT> + </img>
    NUM_IMAGE_TOKEN = 256
    image_tokens = "<img>" + "<IMG_CONTEXT>" * NUM_IMAGE_TOKEN + "</img>"
    prompt = f"{image_tokens}\nHuman: {text_input}\nAssistant: {answer}"
    # OLD: prompt = f"<image>\nHuman: {text_input}\nAssistant: {answer}"

    print(f"Full prompt:\n{prompt}")

    # Tokenize
    tokenized = tokenizer(
        prompt,
        padding=False,
        truncation=True,
        max_length=1024,
        return_tensors='pt',
        add_special_tokens=True
    )

    input_ids = tokenized['input_ids'][0]

    print(f"\nTokenization results:")
    print(f"Input IDs shape: {input_ids.shape}")
    print(f"Input IDs: {input_ids}")

    # Decode tokens to see what they represent
    print(f"\nToken mapping:")
    # OLD: for i, token_id in enumerate(input_ids):
    # OLD:     token = tokenizer.decode([token_id])
    # OLD:     print(f"  {i:3d}: {token_id:5d} -> '{token}'")

    # Create labels (as in dataset)
    labels = input_ids.clone()
    assistant_pos = prompt.find("Assistant:")
    if assistant_pos != -1:
        prefix = prompt[:assistant_pos + len("Assistant:")]
        prefix_tokens = tokenizer(prefix, add_special_tokens=False).input_ids
        if torch.is_tensor(prefix_tokens):
            prefix_tokens = prefix_tokens.tolist()

        prefix_len = 0
        for i in range(min(len(prefix_tokens), len(input_ids))):
            if input_ids[i].item() == prefix_tokens[i]:
                prefix_len = i + 1
            else:
                break

        if prefix_len < len(labels):
            labels[:prefix_len] = -100

    print(f"\nLabels (with -100 for masked positions):")
    print(f"Labels: {labels}")

    # Image flags
    # OLD: image_token_id = tokenizer.convert_tokens_to_ids('<image>')
    # OLD: image_flags = torch.zeros_like(input_ids, dtype=torch.long)
    # OLD: for i, token_id in enumerate(input_ids):
    # OLD:     if token_id == image_token_id:
    # OLD:         image_flags[i] = 1

    # OLD:     print(f"  {i:3d}: {token_id:5d} -> '{token}'")
    # OLD: print(f"Image flags: {image_flags}")

    print(f"\n{'='*80}")
    print(f"✅ SAMPLE ANALYSIS COMPLETE")
    print(f"{'='*80}")


In [36]:
def check_directory_structure():
    """Check the directory structure and available files"""
    print("📁 Checking directory structure...")

    directories_to_check = [
        '.',
        'train',
        'train/videos',
        'public_test',
        'public_test/videos'
    ]

    for dir_path in directories_to_check:
        if os.path.exists(dir_path):
            print(f"\n📂 Contents of '{dir_path}':")
            try:
                items = os.listdir(dir_path)
                for item in items[:10]:  # Show first 10 items
                    full_path = os.path.join(dir_path, item)
                    if os.path.isfile(full_path):
                        size = os.path.getsize(full_path)
                        print(f"   📄 {item} ({size} bytes)")
                    else:
                        print(f"   📁 {item}/")
                if len(items) > 10:
                    print(f"   ... and {len(items) - 10} more items")
            except PermissionError:
                print(f"   ❌ Permission denied")
        else:
            print(f"\n❌ Directory '{dir_path}' does not exist")

In [37]:
check_directory_structure()

📁 Checking directory structure...

📂 Contents of '.':
   📄 .DS_Store (6148 bytes)
   📁 public_test/
   📄 readme.md (1196 bytes)
   📁 train/
   📄 download_videos.py (3231 bytes)
   📄 abc (321 bytes)
   📁 .ipynb_checkpoints/
   📁 vintern-traffic-qa/
   📄 =4.46.0 (18734 bytes)
   📁 vintern-traffic-qa-final/
   ... and 1 more items

📂 Contents of 'train':
   📄 .DS_Store (6148 bytes)
   📁 videos/
   📄 train.json (1014395 bytes)

📂 Contents of 'train/videos':
   📄 17dc8eff_179_clip_008_0051_0060_Y.mp4 (26486619 bytes)
   📄 288b78b5_238_clip_008_0043_0052_Y.mp4 (23196239 bytes)
   📄 d284f54e_554_clip_007_0044_0054_N.mp4 (37523337 bytes)
   📄 7aa4f052_442_clip_007_0047_0054_N.mp4 (21880590 bytes)
   📄 0f02bad0_004_clip_021_0141_0147_Y.mp4 (6102189 bytes)
   📄 ca6fbff8_545_clip_007_0048_0054_Y.mp4 (13530118 bytes)
   📄 cfb87e6f_461_clip_013_0086_0092_Y.mp4 (8721348 bytes)
   📄 895ccd99_033_clip_003_0019_0027_N.mp4 (24459379 bytes)
   📄 a5cb63a2_035_clip_001_0000_0007_Y.mp4 (5387041 bytes)
   📄 

In [ ]:
display_sample_data(0)

🔧 Loading tokenizer...

📊 Loading data...
✅ Loaded 1490 samples from train.json

📋 SAMPLE DATA ANALYSIS - Index 0
📁 Sample ID: train_0001
❓ Question: Nếu xe ô tô đang chạy ở làn ngoài cùng bên phải trong video này thì xe đó chỉ được phép rẽ phải?
🔘 Choices:
   1. A. Đúng
   2. B. Sai
✅ Answer: B. Sai
🎞️ Support frames: [4.427402]
📹 Video path: train/videos/2b840c67_386_clip_002_0008_0018_Y.mp4

🔍 Checking video file...
Relative path: train/videos/2b840c67_386_clip_002_0008_0018_Y.mp4
Full path: train/videos/2b840c67_386_clip_002_0008_0018_Y.mp4
File exists: True
🎥 Video details:
   - Resolution: 2592x1944
   - FPS: 30.00
   - Total frames: 300
   - Duration: 10.00s

🖼️ Extracting frame...
Extracting frame from: train/videos/2b840c67_386_clip_002_0008_0018_Y.mp4
Frame time: 4.427402s
Video info - FPS: 30.00, Total frames: 300, Duration: 10.00s
Target frame index: 132/299
✅ Successfully extracted frame 132

📸 Displaying images...
